# Identification automatique de la langue à partir de la parole
## Approche basée sur les MFCC et les Gaussian Mixture Models (GMM)

Ce notebook implémente une chaîne complète d’identification de langue à partir
de signaux de parole.

### Étapes principales :
- Prétraitement audio et suppression du silence
- Extraction des coefficients MFCC
- Apprentissage de modèles GMM (un par langue)
- Tests sur des segments de durée variable (5s, 10s, 15s)
- Évaluation des performances (accuracy, matrice de confusion)

### Langues étudiées :
- Français
- Arabe
- Anglais
- Allemand
- Japonais

### Données :
- Google FLEURS (Hugging Face `google/fleurs`), téléchargé directement dans Google Colab

In [ ]:
# Installation des dépendances absentes de Google Colab par défaut
# (pandas, numpy, scikit-learn, seaborn, joblib, matplotlib, datasets sont déjà installés dans Colab)
!pip install -q python_speech_features pydub


## Paramètres généraux du projet

Cette section regroupe tous les paramètres globaux :
- chemins des données
- paramètres des modèles GMM
- configuration du parallélisme
- paramètres de suppression du silence

In [ ]:
# ======================================================
# Importation des bibliothèques nécessaires
# ======================================================

import os
import glob
import random
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from joblib import Parallel, delayed
from pydub import AudioSegment
import python_speech_features as psf

from sklearn.mixture import GaussianMixture
from sklearn.metrics import classification_report, confusion_matrix



# Désactiver les warnings pour une sortie plus propre
warnings.filterwarnings("ignore")

# Langues utilisées dans l'expérience
LANGUAGES = ['French', 'Arabic', 'English', 'German', 'Japanese']

# Dossier racine contenant les sous-dossiers des langues
BASE_DIR = '/content/FLEURS_Dataset'

# Dossier de sauvegarde des modèles entraînés
MODEL_DIR = '/content/256_GMM'
# Pour conserver les données/modèles entre deux sessions Colab, monter Google Drive puis utiliser par exemple :
# from google.colab import drive; drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/FLEURS_Dataset'
# MODEL_DIR = '/content/drive/MyDrive/256_GMM'

# Nombre de cœurs CPU utilisés pour le calcul parallèle
N_JOBS = min(8, os.cpu_count() or 1)   # Colab gratuit : 2 cœurs seulement

# Graine pour assurer la reproductibilité
SEED = 42

# Paramètres de suppression du silence
CONFIDENCE_THRESHOLD = 0.99   # Seuil de confiance pour la parole
MIN_SPEECH_FRAMES = 10        # Nombre minimum de frames consécutives

# Paramètres du modèle GMM
N_COMPONENTS = 256

# Quotas de test (durée en secondes : nombre de segments)
TEST_QUOTAS = {
    5: 30,
    10: 15,
    15: 10
}

# --- AJOUT FLEURS : paramètres du dataset Google FLEURS ---
# Correspondance nom de langue du notebook -> configuration FLEURS
FLEURS_CODES = {
    'French':   'fr_fr',
    'Arabic':   'ar_eg',
    'English':  'en_us',
    'German':   'de_de',
    'Japanese': 'ja_jp'
}

# Nombre de fichiers par langue (phase de test du fonctionnement)
N_FILES_PER_LANG = 100         # fichiers d'apprentissage (split FLEURS "train")
N_TEST_FILES_PER_LANG = 100    # fichiers de test (split FLEURS "test")

print(f"Configuration chargée. Utilisation de {N_JOBS} cœurs CPU.")

## Téléchargement et préparation des données (Google FLEURS)

Cette section remplace le dossier de données local d'origine :
- les données sont téléchargées depuis Hugging Face (`google/fleurs`) en mode *streaming* (seuls les premiers fichiers demandés sont lus) ;
- les fichiers sont convertis en WAV 16 bits mono et rangés dans `BASE_DIR/<Langue>/train` et `BASE_DIR/<Langue>/test`, la structure attendue par le reste du notebook ;
- la fréquence d'échantillonnage d'origine de FLEURS (16 kHz) est conservée, sans rééchantillonnage.


In [ ]:
# ======================================================
# Téléchargement et préparation des données Google FLEURS
# ======================================================

import io
from datasets import load_dataset, Audio

# Le code d'origine suppose que MODEL_DIR existe déjà (joblib.dump ne crée pas le dossier)
os.makedirs(MODEL_DIR, exist_ok=True)


def prepare_fleurs_split(lang, fleurs_split, n_files, out_dir):
    """
    Télécharge les n_files premiers enregistrements d'un split FLEURS
    et les enregistre en WAV (16 bits, mono) dans out_dir.

    Paramètres :
    ----------
    lang : str
        Nom de la langue (clé de FLEURS_CODES)
    fleurs_split : str
        Split FLEURS : 'train', 'validation' ou 'test'
    n_files : int
        Nombre de fichiers à conserver
    out_dir : str
        Dossier de destination (BASE_DIR/<Langue>/train ou /test)

    Retour :
    -------
    (int, float, set)
        Nombre de fichiers, durée totale (s), fréquences d'échantillonnage rencontrées
    """
    os.makedirs(out_dir, exist_ok=True)
    code = FLEURS_CODES[lang]

    # Si le dossier est déjà rempli (ré-exécution de la cellule), on ne retélécharge pas
    existing = glob.glob(os.path.join(out_dir, "*.wav"))
    if len(existing) >= n_files:
        print(f"   [+] {fleurs_split}: {len(existing)} fichiers déjà présents dans {out_dir} (téléchargement ignoré)")
        return len(existing), None, set()

    # streaming=True : on ne télécharge que les premières lignes du split demandé
    ds = load_dataset("google/fleurs", code, split=fleurs_split, streaming=True)

    # decode=False : on récupère les octets bruts du fichier audio (pas de décodage automatique),
    # ce qui évite toute dépendance à torchcodec/soundfile selon la version de `datasets`
    ds = ds.cast_column("audio", Audio(decode=False))

    n_saved, total_sec, rates = 0, 0.0, set()
    for ex in ds:
        if n_saved >= n_files:
            break
        try:
            raw = ex["audio"]
            if raw.get("bytes") is not None:
                audio = AudioSegment.from_file(io.BytesIO(raw["bytes"]))
            else:
                audio = AudioSegment.from_file(raw["path"])

            # Format attendu par train_worker : WAV, mono, 16 bits (sample_width == 2)
            if audio.channels > 1:
                audio = audio.set_channels(1)
            if audio.sample_width != 2:
                audio = audio.set_sample_width(2)
        except Exception as e:
            print(f"   [!] Fichier ignoré ({type(e).__name__}: {e})")
            continue

        fname = f"{code}_{fleurs_split}_{n_saved:04d}_id{ex.get('id', n_saved)}.wav"
        audio.export(os.path.join(out_dir, fname), format="wav")

        rates.add(audio.frame_rate)
        total_sec += len(audio) / 1000.0
        n_saved += 1

    return n_saved, total_sec, rates


print("===========================================")
print("   PRÉPARATION DU DATASET GOOGLE FLEURS")
print("===========================================")

for lang in LANGUAGES:
    print(f"\n{lang.upper()} ({FLEURS_CODES[lang]})")

    for fleurs_split, folder, n_files in [("train", "train", N_FILES_PER_LANG),
                                          ("test",  "test",  N_TEST_FILES_PER_LANG)]:
        out_dir = os.path.join(BASE_DIR, lang, folder)
        n_saved, total_sec, rates = prepare_fleurs_split(lang, fleurs_split, n_files, out_dir)
        if total_sec is not None:
            print(f"   > {folder}: {n_saved} fichiers, {total_sec/60:.1f} min, "
                  f"fréquence(s) d'échantillonnage : {sorted(rates)} Hz")

print("\nDonnées prêtes dans :", BASE_DIR)


In [ ]:
# --- FONCTIONS ---

def filter_short_bursts(binary_mask, min_length):
    """
    Supprime les segments très courts (bruit) dans un masque binaire.

    Paramètres :
    ----------
    binary_mask : np.array
        Masque indiquant parole (True) / non-parole (False)
    min_length : int
        Longueur minimale (en frames) d'un segment valide

    Retour :
    -------
    np.array
        Masque nettoyé
    """
    padded = np.concatenate(([0], binary_mask, [0]))
    changes = np.abs(np.diff(padded))
    ranges = np.where(changes == 1)[0].reshape(-1, 2)

    clean_mask = binary_mask.copy()
    for start, end in ranges:
        if (end - start) < min_length:
            clean_mask[start:end] = False

    return clean_mask

In [ ]:
def process_audio_data(samples, sr):
    """
    Pipeline principal de traitement du signal audio :
    - Extraction des MFCC
    - Calcul des coefficients delta
    - Détection et suppression du silence via un mini-GMM
    - Nettoyage des caractéristiques

    Paramètres :
    ----------
    samples : np.array
        Signal audio normalisé
    sr : int
        Fréquence d'échantillonnage

    Retour :
    -------
    np.ndarray
        Matrice de caractéristiques MFCC + Delta nettoyées
    """
    try:
        # Ignorer les signaux trop courts
        if len(samples) < sr * 0.5:
            return None

        # Taille FFT adaptée à la fréquence d'échantillonnage
        n_fft = 1024 if sr <= 16000 else 2048

        # Extraction des MFCC
        mfccs = psf.mfcc(
            samples,
            sr,
            numcep=13,
            nfft=n_fft,
            winfunc=np.hamming,
            appendEnergy=True
        )

        # Calcul des deltas (dérivées temporelles)
        deltas = psf.delta(mfccs, 2)

        # Énergie logarithmique utilisée pour détecter le silence
        log_energy = mfccs[:, 0].reshape(-1, 1)

        # GMM à 2 composantes : parole / non-parole
        gmm_clean = GaussianMixture(n_components=2, random_state=SEED)
        gmm_clean.fit(log_energy)

        speech_idx = np.argmax(gmm_clean.means_.ravel())
        probs = gmm_clean.predict_proba(log_energy)

        # Masque des frames de parole
        mask = probs[:, speech_idx] > CONFIDENCE_THRESHOLD
        clean_mask = filter_short_bursts(mask, MIN_SPEECH_FRAMES)

        # Concaténation MFCC + Delta
        features = np.hstack((mfccs, deltas))
        features_cleaned = features[clean_mask]

        # Sécurité si trop peu de frames restent
        if features_cleaned.shape[0] < 5:
            return features

        return features_cleaned

    except Exception:
        return None

In [ ]:
def train_worker(filepath):
    """
    Fonction utilisée pour l'apprentissage parallèle :
    - Chargement du fichier WAV
    - Conversion en mono
    - Normalisation
    - Extraction des caractéristiques MFCC

    Paramètre :
    ----------
    filepath : str
        Chemin du fichier audio

    Retour :
    -------
    np.ndarray ou None
        Caractéristiques audio extraites
    """
    try:
        audio = AudioSegment.from_wav(filepath)

        # Conversion en mono si nécessaire
        if audio.channels > 1:
            audio = audio.set_channels(1)

        samples = np.array(audio.get_array_of_samples())

        # Normalisation selon la profondeur des bits
        if audio.sample_width == 2:
            samples = samples / 32768.0
        elif audio.sample_width == 4:
            samples = samples / 2147483648.0

        return process_audio_data(samples, audio.frame_rate)

    except Exception:
        return None

In [ ]:
# --- VÉRIFICATION DE COMPATIBILITÉ (optionnelle) ---
# Applique train_worker (inchangé) à quelques fichiers de chaque langue
# et affiche les formats réellement vus par le pipeline.

print(f"{'Langue':<10} {'fichier':<34} {'sr':>6} {'canaux':>6} {'bits':>4} {'durée(s)':>8}  features (frames x dim)")

for lang in LANGUAGES:
    check_files = sorted(glob.glob(os.path.join(BASE_DIR, lang, 'train', '*.wav')))[:3]
    if not check_files:
        print(f"{lang:<10} [!] aucun fichier trouvé dans {os.path.join(BASE_DIR, lang, 'train')}")
        continue

    for f in check_files:
        a = AudioSegment.from_wav(f)
        feats = train_worker(f)
        if feats is None:
            print(f"{lang:<10} {os.path.basename(f):<34} {a.frame_rate:>6} {a.channels:>6} {a.sample_width*8:>4} "
                  f"{len(a)/1000:>8.1f}  None  [!] problème")
        else:
            ok = np.isfinite(feats).all() and feats.ndim == 2 and feats.shape[1] == 26
            print(f"{lang:<10} {os.path.basename(f):<34} {a.frame_rate:>6} {a.channels:>6} {a.sample_width*8:>4} "
                  f"{len(a)/1000:>8.1f}  {feats.shape}  {feats.dtype}  {'OK' if ok else '[!] dimension/valeurs inattendues'}")


## Apprentissage des modèles GMM

Un modèle GMM est entraîné pour chaque langue.
Si le modèle existe déjà sur le disque, l'entraînement est ignoré.

In [ ]:
for lang in LANGUAGES:
    print(f"\nProcessing Language: {lang.upper()}")
    
    # --- MODIFICATION START: Check for existing model ---
    abbr = lang[:2]
    model_filename = os.path.join(MODEL_DIR, f"{abbr}_{N_COMPONENTS}.gmm")
    
    if os.path.exists(model_filename):
        print(f"   [+] Model already exists: {model_filename}")
        print("   > Skipping training for this language.")
        continue
    # --- MODIFICATION END ---
    
    # 1. Locate Files
    path_test = os.path.join(BASE_DIR, lang, 'train')
    path_root = os.path.join(BASE_DIR, lang)
    source_dir = path_test if os.path.exists(path_test) else path_root
    
    if not os.path.exists(source_dir):
        print(f"   [!] Directory not found for {lang}. Skipping.")
        continue

    train_files = glob.glob(os.path.join(source_dir, "*.wav"))
    
    
    print(f"   > Found {len(train_files)} files. Training on {len(train_files)}.")
    
    # 3. Parallel Extraction
    print(f"   > Extracting features with {N_JOBS} cores...")
    results = Parallel(n_jobs=N_JOBS)(delayed(train_worker)(f) for f in train_files)
    valid_features = [res for res in results if res is not None]
    
    if not valid_features:
        print("   [!] No valid features. Skipping.")
        continue

    X_train = np.vstack(valid_features)
    print(f"   > Training Matrix: {X_train.shape}")

    # 4. Train GMM
    print(f"   > Training GMM with {N_COMPONENTS} components...")
    gmm = GaussianMixture(n_components=N_COMPONENTS, random_state=SEED, n_init=3)
    gmm.fit(X_train)
    
    # 5. Save (Filename logic is now handled at start of loop, but we use the variable here)
    joblib.dump(gmm, model_filename)
    print(f"   > Model saved: {model_filename}")

## Phase de test

Les fichiers audio de test sont découpés en segments de :
- 5 secondes
- 10 secondes
- 15 secondes

Chaque segment est évalué par tous les modèles GMM.
La langue prédite correspond au score de vraisemblance maximal.

In [ ]:


# --- INDEPENDENT TESTING PIPELINE (SAVES AUDIO & DETAILED SCORES) ---
import os
import glob
import random
import joblib
import numpy as np
import pandas as pd
from pydub import AudioSegment
from sklearn.model_selection import train_test_split

print("===========================================")
print(f"   STARTING DETAILED TEST (GMM-{N_COMPONENTS})")
print("===========================================")


# --- SETUP OUTPUT FOLDERS ---
output_dir = "Experiment_Results"
audio_save_dir = os.path.join(output_dir, "Test_Clips")
os.makedirs(audio_save_dir, exist_ok=True)
print(f"   [+] Audio clips will be saved to: {audio_save_dir}")

# 1. Load Models from Disk
models = {}
print("1. Loading Models...")

for lang in LANGUAGES:
    abbr = lang[:2]
    model_filename = f"{abbr}_{N_COMPONENTS}.gmm"
    model_path = os.path.join(MODEL_DIR, model_filename)
    
    if os.path.exists(model_path):
        models[lang] = joblib.load(model_path)
        print(f"   [+] Loaded: {model_filename}")
    else:
        print(f"   [!] Missing: {model_filename} (Skipping {lang})")

if not models:
    print("\nError: No models found. Please run training first.")
else:
    results_data = []

    # 2. Test Loop
    for true_lang in LANGUAGES:
        print(f"\n2. Testing Language: {true_lang}")
        
        # Locate files
        path_test = os.path.join(BASE_DIR, true_lang, 'test')
        path_root = os.path.join(BASE_DIR, true_lang)
        source_dir = path_test if os.path.exists(path_test) else path_root
        
        if not os.path.exists(source_dir):
            print(f"   [!] Folder not found: {source_dir}")
            continue

        test_files = glob.glob(os.path.join(source_dir, "*.wav"))
        
        
        quotas_filled = {5: 0, 10: 0, 15: 0}
        
        for filepath in test_files:
            if all(quotas_filled[d] >= TEST_QUOTAS[d] for d in TEST_QUOTAS):
                break
                
            try:
                audio = AudioSegment.from_wav(filepath)
                if audio.channels > 1: audio = audio.set_channels(1)
            except:
                continue
                
            current_pos = 0
            
            for duration in [15, 10, 5]:
                if quotas_filled[duration] < TEST_QUOTAS[duration]:
                    
                    if (len(audio) - current_pos) >= (duration * 1000):
                        
                        # Cut Clip
                        clip = audio[current_pos : current_pos + (duration * 1000)]
                        current_pos += (duration * 1000)
                        
                        # Generate Name: Language_Filename_Duration_Index
                        clip_idx = quotas_filled[duration] + 1
                        fname_base = os.path.basename(filepath).replace('.wav', '')
                        segment_name = f"{true_lang}_{fname_base}_{duration}s_{clip_idx}"
                        
                        # --- SAVE AUDIO CLIP TO DISK ---
                        save_path = os.path.join(audio_save_dir, segment_name + ".wav")
                        clip.export(save_path, format="wav")
                        
                        # Process
                        samples = np.array(clip.get_array_of_samples())
                        if clip.sample_width == 2: samples = samples / 32768.0
                        elif clip.sample_width == 4: samples = samples / 2147483648.0
                        
                        features = process_audio_data(samples, clip.frame_rate)
                        
                        if features is not None:
                            # --- SCORING ALL MODELS ---
                            clip_scores = []
                            
                            # 1. Get scores
                            for m_lang, model in models.items():
                                try:
                                    score = model.score(features)
                                    clip_scores.append((m_lang, score))
                                except:
                                    clip_scores.append((m_lang, -float('inf')))
                            
                            # 2. Determine Winner
                            clip_scores.sort(key=lambda x: x[1], reverse=True)
                            winner_lang = clip_scores[0][0]
                            
                            # 3. Save Row PER MODEL
                            for m_lang, score in clip_scores:
                                eval_model_name = f"{m_lang[:2]}_{N_COMPONENTS}.gmm"
                                
                                results_data.append({
                                    "Model": eval_model_name,
                                    "segment": segment_name,  # Matches the saved filename
                                    "score": score,
                                    "decision": winner_lang,
                                    "Actual": true_lang,
                                    "Duration": duration
                                })
                            
                            quotas_filled[duration] += 1
        
        print(f"   > Quotas: 5s:{quotas_filled[5]} | 10s:{quotas_filled[10]} | 15s:{quotas_filled[15]}")

    print("\nTesting Complete.")

## Évaluation des performances

- Accuracy globale
- Rapport de classification
- Matrice de confusion normalisée
- Analyse par durée de segment

In [ ]:
# --- ANALYSIS & SAVE RESULTS ---

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Setup Folder (Same as above)
output_dir = "Experiment_Results"
os.makedirs(output_dir, exist_ok=True)

# 2. Create DataFrame
df_results = pd.DataFrame(results_data)

if not df_results.empty:
    
    # --- SAVE RAW CSV (Detailed: 1 row per model per file) ---
    csv_columns = ['Model', 'segment', 'score', 'decision']
    csv_path = os.path.join(output_dir, f"test_results_{N_COMPONENTS}.csv")
    df_results[csv_columns].to_csv(csv_path, index=False)
    print(f"[+] Saved Detailed CSV results to: {csv_path}")

    # --- PREPARE DATA FOR METRICS (Deduplicate) ---
    # We drop duplicates to calculate accuracy based on distinct segments
    df_metrics = df_results.drop_duplicates(subset=['segment'])
    
    # Calculate Correctness
    df_metrics['Correct'] = df_metrics['Actual'] == df_metrics['decision']
    acc = df_metrics['Correct'].mean() * 100

    print("===========================================")
    print(f"             FINAL REPORT (GMM-{N_COMPONENTS})")
    print("===========================================")
    print(f"Overall Accuracy: {acc:.2f}%")
    
    # --- SAVE SUMMARY REPORT ---
    txt_path = os.path.join(output_dir, f"Performance_Report_{N_COMPONENTS}_GMM.txt")
    with open(txt_path, "w") as f:
        f.write("===========================================\n")
        f.write(f"      LANGUAGE IDENTIFICATION REPORT       \n")
        f.write(f"      Model: GMM ({N_COMPONENTS} components)\n")
        f.write("===========================================\n\n")
        
        f.write(f"Overall Accuracy: {acc:.2f}%\n\n")
        
        f.write("--- Classification Report ---\n")
        f.write(classification_report(df_metrics['Actual'], df_metrics['decision']))
        
        # Accuracy by Duration
        dur_pivot = df_metrics.pivot_table(index='Actual', columns='Duration', values='Correct', aggfunc='mean') * 100
        f.write("\n\n--- Accuracy by Duration ---\n")
        f.write(str(dur_pivot.round(1)))
        
    print(f"[+] Saved summary report to: {txt_path}")

    # --- SAVE CONFUSION MATRIX ---
    cm = confusion_matrix(df_metrics['Actual'], df_metrics['decision'], labels=LANGUAGES)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=LANGUAGES, yticklabels=LANGUAGES)
    plt.title(f'Confusion Matrix (GMM-{N_COMPONENTS} | Acc: {acc:.2f}%)')
    plt.ylabel('True Language')
    plt.xlabel('Predicted Language')
    
    img_path = os.path.join(output_dir, f"Confusion_Matrix_{N_COMPONENTS}_GMM.png")
    plt.savefig(img_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"[+] Saved confusion matrix to: {img_path}")

else:
    print("No results found. Please run the testing pipeline first.")

## Pipeline complet : détection de langue → transcription → traduction → synthèse vocale

Cette section s'appuie sur le reste du notebook (modèles GMM, `train_worker`, `process_audio_data`) :

1. **Détection de la langue** : les 5 modèles GMM entraînés plus haut (même scoring que la phase de test).
2. **Transcription (ASR)** : Whisper, avec la langue détectée à l'étape 1.
3. **Traduction** : NLLB-200, de la langue détectée vers la langue cible choisie.
4. **Synthèse vocale (TTS)** : gTTS, dans la langue cible.


In [ ]:
# Dépendances du pipeline (torch, numpy, pydub sont déjà dans Colab)
!pip install -q transformers sentencepiece gTTS


In [ ]:
# ======================================================
# PIPELINE COMPLET : détection -> transcription -> traduction -> synthèse
# ======================================================
import tempfile
from IPython.display import Audio, display

WHISPER_MODEL = "openai/whisper-small"                 # ASR (plus précis : whisper-medium / large-v3, plus lents)
NLLB_MODEL = "facebook/nllb-200-distilled-600M"        # traduction

# Codes de langue propres à chaque outil
LANG_CODES = {
    'French':   {'whisper': 'french',   'nllb': 'fra_Latn', 'gtts': 'fr'},
    'Arabic':   {'whisper': 'arabic',   'nllb': 'arb_Arab', 'gtts': 'ar'},
    'English':  {'whisper': 'english',  'nllb': 'eng_Latn', 'gtts': 'en'},
    'German':   {'whisper': 'german',   'nllb': 'deu_Latn', 'gtts': 'de'},
    'Japanese': {'whisper': 'japanese', 'nllb': 'jpn_Jpan', 'gtts': 'ja'},
}

_CACHE = {}   # les gros modèles ne sont chargés qu'une fois


def to_wav_16k(audio_path):
    """Convertit n'importe quel audio (wav, mp3, m4a...) en WAV 16 kHz mono 16 bits (format de l'entraînement)."""
    audio = AudioSegment.from_file(audio_path).set_channels(1).set_frame_rate(16000).set_sample_width(2)
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    audio.export(tmp.name, format="wav")
    return tmp.name


def detect_language_gmm(audio_path):
    """Étape 1 : détection de la langue avec les GMM entraînés (même scoring que la phase de test)."""
    if 'gmm_models' not in _CACHE:
        _CACHE['gmm_models'] = {}
        for lang in LANGUAGES:
            path = os.path.join(MODEL_DIR, f"{lang[:2]}_{N_COMPONENTS}.gmm")
            if os.path.exists(path):
                _CACHE['gmm_models'][lang] = joblib.load(path)
    gmm_models = _CACHE['gmm_models']
    if not gmm_models:
        raise RuntimeError("Aucun modèle GMM trouvé : exécute d'abord la section d'apprentissage.")

    features = train_worker(to_wav_16k(audio_path))
    if features is None:
        raise ValueError("Audio illisible ou trop court (< 0,5 s).")

    scores = {lang: model.score(features) for lang, model in gmm_models.items()}
    return max(scores, key=scores.get), scores


def transcribe_audio(audio_path, language):
    """Étape 2 : transcription avec Whisper, en lui imposant la langue détectée."""
    if 'asr' not in _CACHE:
        import torch
        from transformers import pipeline
        _CACHE['asr'] = pipeline("automatic-speech-recognition", model=WHISPER_MODEL,
                                 chunk_length_s=30, device=0 if torch.cuda.is_available() else -1)
    audio = AudioSegment.from_file(to_wav_16k(audio_path))
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    out = _CACHE['asr']({"raw": samples, "sampling_rate": 16000},
                        generate_kwargs={"language": LANG_CODES[language]['whisper'], "task": "transcribe"})
    return out["text"].strip()


def translate_text(text, source_language, target_language):
    """Étape 3 : traduction avec NLLB-200."""
    if 'nllb' not in _CACHE:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        _CACHE['nllb_tok'] = AutoTokenizer.from_pretrained(NLLB_MODEL)
        _CACHE['nllb'] = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
        if torch.cuda.is_available():
            _CACHE['nllb'] = _CACHE['nllb'].to("cuda")
    tok, model = _CACHE['nllb_tok'], _CACHE['nllb']
    tok.src_lang = LANG_CODES[source_language]['nllb']
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    out = model.generate(**inputs,
                         forced_bos_token_id=tok.convert_tokens_to_ids(LANG_CODES[target_language]['nllb']),
                         max_new_tokens=400)
    return tok.batch_decode(out, skip_special_tokens=True)[0]


def synthesize_speech(text, target_language, out_path="speech_output.mp3"):
    """Étape 4 : synthèse vocale avec gTTS (nécessite Internet)."""
    from gtts import gTTS
    gTTS(text=text, lang=LANG_CODES[target_language]['gtts']).save(out_path)
    return out_path


def pipeline_complet(audio_path, target_language, source_language=None, out_path="speech_output.mp3"):
    """
    audio -> langue détectée -> texte -> texte traduit -> audio de la langue cible.
    source_language : à renseigner pour forcer la langue (saute la détection GMM).
    """
    if target_language not in LANG_CODES:
        raise ValueError(f"Langue cible inconnue : {target_language}. Choix : {list(LANG_CODES)}")

    print("1. Détection de la langue (GMM)...")
    if source_language is None:
        source_language, scores = detect_language_gmm(audio_path)
        for lang, s in sorted(scores.items(), key=lambda x: x[1], reverse=True):
            print(f"     {lang:<9} score = {s:.3f}")
    print(f"   > Langue détectée : {source_language}")

    print("2. Transcription (Whisper)...")
    text = transcribe_audio(audio_path, source_language)
    print(f"   > Texte : {text}")

    if source_language == target_language:
        print("3. Traduction ignorée (langue source = langue cible).")
        translated = text
    else:
        print(f"3. Traduction (NLLB) : {source_language} -> {target_language}...")
        translated = translate_text(text, source_language, target_language)
        print(f"   > Traduction : {translated}")

    print(f"4. Synthèse vocale (gTTS, {target_language})...")
    out_file = synthesize_speech(translated, target_language, out_path)
    print(f"   > Audio enregistré : {out_file}")

    return {"source_language": source_language, "text": text,
            "translated_text": translated, "audio_file": out_file}


In [ ]:
# --- DÉMONSTRATION ---
# Choisis un fichier audio (ici : un fichier de test FLEURS) et la langue cible
TARGET_LANGUAGE = 'English'      # French / Arabic / English / German / Japanese

demo_files = sorted(glob.glob(os.path.join(BASE_DIR, 'French', 'test', '*.wav')))
audio_path = demo_files[0]       # ou ton propre fichier : audio_path = '/content/mon_audio.wav'
# Attention : les GMM sont entraînés sur du WAV propre. Un mp3/m4a fortement compressé dégrade les scores
# et peut faire se tromper la détection : préfère un WAV, ou force la langue avec source_language='French'.

print("Audio d'entrée :")
display(Audio(audio_path))

result = pipeline_complet(audio_path, TARGET_LANGUAGE)

print("\nAudio de sortie :")
display(Audio(result["audio_file"]))


## Conclusion

Ce notebook présente un système complet d'identification de langue
basé sur des méthodes statistiques classiques (GMM).

Points forts :
- Suppression automatique du silence
- Apprentissage parallèle
- Évaluation détaillée
- Pipeline reproductible

Perspectives :
- Ajout d'autres langues
- Utilisation de i-vectors / x-vectors
- Approches Deep Learning